# Run MAT -> NPZ Export Without Console

This notebook is a simple UI-like wrapper for the converters.

How to use:
1. Run cells from top to bottom.
2. Edit only the path variables in the config cells.
3. Execute the run cells.

It supports:
- `mat_to_npz_structs.py` (exports all top-level structs from one `*_Ratemap*.mat`)

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return p


def resolve_path(path_str: str, root: Path) -> Path:
    p = Path(path_str)
    return p if p.is_absolute() else (root / p)


def run_python_script(repo_root: Path, script_rel: str, args: list[str]) -> subprocess.CompletedProcess[str]:
    script_path = (repo_root / script_rel).resolve()
    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")

    cmd = [sys.executable, str(script_path), *args]
    print("Running command:")
    print(" ".join(shlex.quote(c) for c in cmd))

    result = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
    print("\nSTDOUT:\n" + (result.stdout or "<empty>"))
    if result.stderr:
        print("\nSTDERR:\n" + result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Script failed with code {result.returncode}")
    return result


REPO_ROOT = find_repo_root()
print(f"Repo root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")

## 1) Configure Struct Export (`mat_to_npz_structs.py`)

Edit these values, then run the next cell.

In [ ]:
INPUT_RATEMAP_MAT = r"data\raw\VS98_2024-05-15_17-29-02_Ratemap_final_thèse.mat"  # one *_Ratemap*.mat file
OUTPUT_STRUCT_DIR = r"path\\VS98_2024-05-15_17-29-02_Ratemap_final_thèse"    # output folder to create/use
OVERWRITE_STRUCT_EXPORT = True  # whether to overwrite existing output files

In [ ]:
input_mat = resolve_path(INPUT_RATEMAP_MAT, REPO_ROOT)
output_dir = resolve_path(OUTPUT_STRUCT_DIR, REPO_ROOT)

if not input_mat.exists():
    raise FileNotFoundError(f"Input not found: {input_mat}")
if input_mat.suffix.lower() != ".mat":
    raise ValueError(f"Input must be .mat: {input_mat}")
if "ratemap" not in input_mat.name.lower():
    raise ValueError(f"Expected '*_Ratemap*.mat', got: {input_mat.name}")

args = ["--input", str(input_mat), "--output", str(output_dir)]
if OVERWRITE_STRUCT_EXPORT:
    args.append("--overwrite")

run_python_script(REPO_ROOT, "mat_to_npz_structs.py", args)

print("\nGenerated files:")
if output_dir.exists():
    files = sorted(output_dir.glob(f"{input_mat.stem}_*.npz"))
    if not files:
        print(" - No matching output files found yet.")
    for p in files:
        print(" -", p)
